# 📊 Модуль 6. Урок 1: Подключение источников данных и архитектура витрин в Looker Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_06_bi_looker_studio/lesson_01_looker_studio_setup_and_data_sources.ipynb)

---

## 1. Ментальная модель BI: Измерения (Dimensions) vs Показатели (Metrics)

В Looker Studio и любом современном BI-инструменте данные жестко разделяются:
- **Dimensions (Зеленые поля)**: Категориальные срезы, по которым данные группируются (Город, Категория, Дата, Канал привлечения).
- **Metrics (Синие поля)**: Числовые агрегируемые величины (Выручка, Количество клиентов, Средний чек).

### Золотое правило витрины данных для BI:
Никогда не подключайте «сырые» ненормализованные логи напрямую в отчеты, если в них миллионы строк. Сначала подготовьте денормализованную витрину (Data Mart) на уровне SQL/Python.


In [ ]:
# Экспорт оптимизированной витрины для Looker Studio
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query_bi = """
SELECT 
    SUBSTR(o.order_date, 1, 10) AS order_date,
    c.city,
    c.acquisition_channel,
    c.device,
    p.category,
    p.product_name,
    oi.quantity,
    oi.unit_price,
    oi.total_amount AS revenue,
    (oi.total_amount - (p.cost * oi.quantity)) AS gross_profit
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status = 'Completed';
"""

bi_data = pd.read_sql(query_bi, conn)
bi_data.to_csv(r"d:\vibe_app\main_data_analytics\course\datasets\processed\looker_studio_source.csv", index=False)
print(" ВИТРИНА ДЛЯ LOOKER STUDIO ГОТОВА:")
print(f"Строк: {len(bi_data):,} | Колонок: {len(bi_data.columns)}")
print(bi_data.head(3))
